# Day 3 homework — add search (project)

Same ideas as **`course/Day_03_Add_Search.ipynb`**, but:

- **Chunked corpus**: `pydantic/pydantic` (not Evidently).  
- **FAQ subset**: DataTalks FAQ filtered by **`machine-learning`** (course uses **`data-engineering`**).

```bash
uv add minsearch sentence-transformers numpy tqdm requests python-frontmatter
uv add --dev jupyter
uv run jupyter notebook
```

In [1]:
import io
import zipfile

import frontmatter
import requests


def read_repo_data(repo_owner, repo_name):
    url = f"https://codeload.github.com/{repo_owner}/{repo_name}/zip/refs/heads/main"
    resp = requests.get(url)
    if resp.status_code != 200:
        raise RuntimeError(f"Failed to download repository: {resp.status_code}")
    out = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    for file_info in zf.infolist():
        fn = file_info.filename
        fnl = fn.lower()
        if not (fnl.endswith(".md") or fnl.endswith(".mdx")):
            continue
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode("utf-8", errors="ignore")
                post = frontmatter.loads(content)
                data = post.to_dict()
                data["filename"] = fn
                out.append(data)
        except Exception as e:
            print(f"skip {fn}: {e}")
    zf.close()
    return out


def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")
    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i : i + size]
        result.append({"start": i, "chunk": chunk})
        if i + size >= n:
            break
    return result

## Chunked Pydantic docs (lexical + vector targets)

Use a smaller embed limit while iterating if CPU is slow.

In [2]:
pydantic_docs = read_repo_data("pydantic", "pydantic")

pydantic_chunks = []
for doc in pydantic_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop("content")
    for chunk in sliding_window(doc_content, 2000, 1000):
        chunk.update(doc_copy)
        pydantic_chunks.append(chunk)

len(pydantic_docs), len(pydantic_chunks)

(98, 1035)

In [3]:
from minsearch import Index

pydantic_text_index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[],
)
pydantic_text_index.fit(pydantic_chunks)

pydantic_text_index.search("How does field validation work?", num_results=3)

[{'start': 12000,
  'chunk': 'dantic import BaseModel, TypeAdapter\n\n\nclass MyModel(BaseModel):\n    a: int\n    b: Annotated[str, MinLen(5)]\n\n\nta = TypeAdapter(list[MyModel])\nv = ta.validate_json(\n    \'[{"a": 1, "b": "12345"}, {"a": 1,\',\n    experimental_allow_partial=True,\n)\nprint(v)\n#> [MyModel(a=1, b=\'12345\')]\n```\n\n### Limitations of Partial Validation\n\n#### TypeAdapter only\n\nYou can only pass `experiment_allow_partial` to [`TypeAdapter`][pydantic.TypeAdapter] methods, it\'s not yet supported via other Pydantic entry points like [`BaseModel`][pydantic.BaseModel].\n\n#### Types supported\n\nRight now only a subset of collection validators know how to handle partial validation:\n\n* `list`\n* `set`\n* `frozenset`\n* `dict` (as in `dict[X, Y]`)\n* `TypedDict` — only non-required fields may be missing, e.g. via [`NotRequired`][typing.NotRequired] or [`total=False`][typing.TypedDict.__total__])\n\nWhile you can use `experimental_allow_partial` while validating agai

## FAQ subset — machine learning track (different filter than course)

Course Day 3 filters **`data-engineering`**. Here we use **`machine-learning`**.

In [4]:
dtc_faq = read_repo_data("DataTalksClub", "faq")
ml_faq = [
    d
    for d in dtc_faq
    if "machine-learning" in (d.get("filename") or "").lower()
]

faq_index = Index(text_fields=["question", "content"], keyword_fields=[])
faq_index.fit(ml_faq)
len(ml_faq)

440

## Vector search + hybrid (FAQ)

First model download can take a minute.

In [5]:
from tqdm.auto import tqdm
import numpy as np
from sentence_transformers import SentenceTransformer
from minsearch import VectorSearch

embedding_model = SentenceTransformer("multi-qa-distilbert-cos-v1")

faq_embeddings = np.array(
    [
        embedding_model.encode(d["question"] + " " + d["content"])
        for d in tqdm(ml_faq)
    ]
)

faq_vindex = VectorSearch()
faq_vindex.fit(faq_embeddings, ml_faq)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/440 [00:00<?, ?it/s]

In [6]:
def text_search(query, num_results=5):
    return faq_index.search(query, num_results=num_results)


def vector_search(query, num_results=5):
    q = embedding_model.encode(query)
    return faq_vindex.search(q, num_results=num_results)


def hybrid_search(query, num_results=5):
    tr = text_search(query, num_results=num_results)
    vr = vector_search(query, num_results=num_results)
    seen = set()
    out = []
    for result in tr + vr:
        key = result.get("id") or result.get("filename")
        if key is None:
            key = (result.get("question", "")[:120], result.get("content", "")[:120])
        if key in seen:
            continue
        seen.add(key)
        out.append(result)
    return out


hybrid_search("Can I still join the course after it starts?", num_results=5)

[{'id': '41aabbd7c5',
  'question': 'The course has already started. Can I still join it?',
  'sort_order': 14,
  'content': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'filename': 'faq-main/_questions/machine-learning-zoomcamp/general/014_41aabbd7c5_the-course-has-already-started-can-i-still-join-it.md'},
 {'id': 'a11770e750',
  'question': 'I’m new to Slack and can’t find the course channel. Where is it?',
  'sort_order': 13,
  'content': "Here’s how you join in Slack: [https://slack.com/help/articles/205239967-Join-a-channel](https://slack.com/help/articles/205239967-Join-a-channel)\n\n- Cli

## Optional: vector index on Pydantic chunks

Embed the **`chunk`** field only (same as Evidently in the course).

In [7]:
MAX_PYDANTIC_EMBED = 150  # raise or set None for all chunks (slow on CPU)

_rows = (
    pydantic_chunks
    if MAX_PYDANTIC_EMBED is None
    else pydantic_chunks[:MAX_PYDANTIC_EMBED]
)

pydantic_embeddings = np.array(
    [embedding_model.encode(d["chunk"]) for d in tqdm(_rows)]
)

pydantic_vindex = VectorSearch()
pydantic_vindex.fit(pydantic_embeddings, _rows)

q = embedding_model.encode("BaseModel config")
pydantic_vindex.search(q, num_results=3)

  0%|          | 0/150 [00:00<?, ?it/s]

[{'start': 76000,
  'chunk': "hub.com/pydantic/pydantic/pull/10518)\n* Fix `stacklevel` on deprecation warnings for `BaseModel` by @sydney-runkle in [#10520](https://github.com/pydantic/pydantic/pull/10520)\n* Fix warning `stacklevel` in `BaseModel.__init__` by @Viicos in [#10526](https://github.com/pydantic/pydantic/pull/10526)\n* Improve error handling for in-evaluable refs for discriminator application by @sydney-runkle in [#10440](https://github.com/pydantic/pydantic/pull/10440)\n* Change the signature of `ConfigWrapper.core_config` to take the title directly by @Viicos in [#10562](https://github.com/pydantic/pydantic/pull/10562)\n* Do not use the previous config from the stack for dataclasses without config by @Viicos in [#10576](https://github.com/pydantic/pydantic/pull/10576)\n* Fix serialization for IP types with `mode='python'` by @sydney-runkle in [#10594](https://github.com/pydantic/pydantic/pull/10594)\n* Support constraint application for `Base64Etc` types by @sydney-runkl

## Homework checklist

- Index **your** Day 1 / Day 2 data.  
- Compare text vs vector vs hybrid; read results critically.  
- Post about the build.

[Course](https://alexeygrigorev.com/aihero/) · [Slack `#course-ai-hero`](https://datatalks.club/)